# ModernBERT-Ja 30M：通常対話用16分類・578件

1. 「ランタイム」→「ランタイムのタイプを変更」→GPUを選択。
2. 「ランタイム」→「すべてのセルを実行」。教材と学習コードは同梱済みです。
3. 3シードの分類別結果・誤分類・devの推移を確認します。
4. 最後に `modernbert-normal578-results.zip` をMacへ保存してからGPUを解放してください。

**train 482 / dev 46 / test 50**。モブ情報・剣・乗用動物・ドギド呼び出し・視線11種・対象外の16分類です。
前版546件を保ち、今回採用した道具・雑談の合成例32件（剣依頼8、対象外24）をtrainへ追加しました。
既存の実機フィードバック15件もtrainに含みます。dev/testと分類IDは前版から変更していません。

前回と同じ seed42・43・44、学習率5e-5、各10 epoch、batch16、128 token、履歴なし、float32/SDPA、TF32無効。
固定した基盤モデルから新しい16分類headを作り、全パラメータを学習します。各seedのepochはdevで選び、全選択の確定後にtestを診断します。

このノートブックは通常対話用の学習と結果保存を行います。川柳用との切替・本体への接続は別作業です。
dev/testは前回の教材設計で参照済みです。実機15件・合成32件の結果は学習内の確認で、未知の実機精度とは区別します。


In [ ]:
# 教材と前回の学習コードを展開（別ZIPのアップロードは不要）
from pathlib import Path
import base64, hashlib, io, json, zipfile
ROOT = Path('/content/modernbert-normal578')
RELEASE_PATH = 'dev_tools/request_classifier/releases/2026-10-24-normal-dialogue-578'
assert not ROOT.exists(), '前回の出力があります。新しいランタイムで実行してください。'
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'cfc326ae0a3942df3626dc4fbfb3140edc58f73e18c03c6edfb14c4206490118'
ROOT.mkdir()
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(ROOT)
for name, expected in json.loads((ROOT/'input-sha256.json').read_text()).items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
print('INPUT VERIFIED: 578件 / 16分類 / train 482, dev 46, test 50')
del payload
from IPython.display import display, Markdown
display(Markdown((ROOT/RELEASE_PATH/'README.md').read_text()))


## GPUと固定環境の準備

In [ ]:
import subprocess, sys, time, platform
setup_start = time.perf_counter()
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11.14', str(ROOT/'.venv')], check=True)
PYTHON = str(ROOT/'.venv/bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON, '-r', str(ROOT/'dev_tools/request_classifier/requirements-train.txt')], check=True)
subprocess.run([PYTHON, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); print(torch.cuda.get_device_properties(0).total_memory); print(torch.__version__, torch.version.cuda)"], check=True)
subprocess.run([PYTHON, '-c', "from huggingface_hub import snapshot_download; snapshot_download('sbintuitions/modernbert-ja-30m', revision='8cb03f54cb9e30e72459e5f1cedc6d89c7d8dcb5', token=False, cache_dir='/content/modernbert-normal578/hf-cache', allow_patterns=['config.json','model.safetensors','tokenizer.json','tokenizer_config.json','special_tokens_map.json','tokenizer.model','spiece.model','README.md','LICENSE'])"], check=True)
(ROOT/'setup.json').write_text(json.dumps({'setup_seconds': time.perf_counter()-setup_start, 'notebook_python': sys.version}, indent=2))
(ROOT/'pip-freeze.txt').write_bytes(subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PYTHON]))
print('SETUP VERIFIED: GPUの名前と容量を確認してください')


## 3シードを順番に学習・選択後に評価

In [ ]:
cmd = [PYTHON, '-u', str(ROOT/'dev_tools/request_classifier/train.py'),
    '--material-release', '--data-dir', str(ROOT/'dev_tools/request_classifier/releases/2026-10-24-normal-dialogue-578'),
    '--data-report', str(ROOT/'dev_tools/request_classifier/releases/2026-10-24-normal-dialogue-578/data_report.json'),
    '--model', 'sbintuitions/modernbert-ja-30m', '--revision', '8cb03f54cb9e30e72459e5f1cedc6d89c7d8dcb5',
    '--seeds', '42', '43', '44', '--learning-rates', '5e-5', '--epochs', '10', '--batch-size', '16',
    '--max-length', '128', '--max-context', '0', '--device', 'cuda',
    '--source-git-head', 'f8e1d0296068dfcd9ab583cb5604b9e0cfe06701',
    '--output-dir', str(ROOT/'weights'), '--report-dir', str(ROOT/'reports')]
(ROOT/'command.json').write_text(json.dumps(cmd, ensure_ascii=False, indent=2))
with (ROOT/'training.log').open('w') as log:
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line); log.flush()
        print(line, end='', flush=True)
    status = proc.wait()
assert status == 0, '学習が停止しました。training.logを確認してください。'
print('TRAINING AND KNOWN-DATA DIAGNOSTIC COMPLETE')


## 分類ごとの結果

In [ ]:
from IPython.display import display, Markdown
manifest = json.loads((ROOT/'reports/manifest.json').read_text())
assert manifest.get('finished_at_utc') and manifest['label_to_id'] == json.loads((ROOT/RELEASE_PATH/'labels.json').read_text())
selection = json.loads((ROOT/'reports/selection.json').read_text())
import sys
sys.path.insert(0, str(ROOT/'dev_tools/request_classifier'))
from evaluation import distribution
seeds = [42, 43, 44]
assert manifest['seeds'] == seeds
assert [item['seed'] for item in selection['selected_models']] == seeds
all_metrics = {seed: json.loads((ROOT/'reports'/f'seed-{seed}'/'metrics.json').read_text()) for seed in seeds}
summary = {'seeds': seeds, 'splits': {}}
for split in ('train', 'dev', 'test'):
    summary['splits'][split] = {key: distribution([all_metrics[seed][split][key] for seed in seeds])
                               for key in ('correct', 'accuracy', 'macro_f1')}
    summary['splits'][split]['per_class'] = {label: {
        'support': all_metrics[seeds[0]][split]['per_class'][label]['support'],
        **{key: distribution([all_metrics[seed][split]['per_class'][label][key] for seed in seeds])
           for key in ('precision', 'recall', 'f1')}}
        for label in manifest['label_to_id']}
(ROOT/'reports/seed-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + '\n')
summary_lines = ['# 3シードの結果', '',
    '同じ482/46/50件の固定分割で、基盤モデルから独立に学習。epochは各seedのdevで選択し、3モデルすべてをtestで診断します。', '',
    '| seed | 選択epoch | dev 正解/対象 | dev macro F1 | test 正解/対象 | test macro F1 |',
    '|---:|---:|---:|---:|---:|---:|']
for selected in selection['selected_models']:
    seed = selected['seed']
    dev, test = all_metrics[seed]['dev'], all_metrics[seed]['test']
    summary_lines.append(f"| {seed} | {selected['epoch']:g} | {dev['correct']}/{dev['n']} | {dev['macro_f1']:.3f} | {test['correct']}/{test['n']} | {test['macro_f1']:.3f} |")
summary_lines += ['', '## 平均と標本標準偏差（3シード）', '',
    '| 分割 | accuracy 平均 ± SD | macro F1 平均 ± SD |', '|---|---:|---:|']
for split, values in summary['splits'].items():
    acc, f1 = values['accuracy'], values['macro_f1']
    summary_lines.append(f"| {split} | {acc['mean']:.2%} ± {acc['sample_stddev']:.2%} | {f1['mean']:.4f} ± {f1['sample_stddev']:.4f} |")
summary_lines += ['', '全シードを記録します。testで最良seedを選ぶ処理や、アンサンブル評価は行いません。',
    'dev/testは教材設計時に参照済みの診断です。SDは固定分割でのseed間のばらつきで、未知入力の精度の信頼区間ではありません。']
summary_text = '\n'.join(summary_lines)
(ROOT/'reports/seed-summary.md').write_text(summary_text + '\n')
display(Markdown(summary_text))
for selected in selection['selected_models']:
    report_dir = ROOT/'reports'/f"seed-{selected['seed']}"
    metrics = all_metrics[selected['seed']]
    display(Markdown(f"# seed {selected['seed']}：分類別・誤分類・dev推移"))
    result = metrics['test']
    lines = ['# 最終診断：test（重み更新・epoch選択に未使用／教材改善時には参照済み）', '',
        f"正解 {result['correct']}/{result['n']}（{result['accuracy']:.1%}） / macro F1 {result['macro_f1']:.3f}", '',
        '| 分類 | 正解/対象 | 適合率 | 再現率 | F1 |', '|---|---:|---:|---:|---:|']
    for i, label in enumerate(result['label_order']):
        row = result['per_class'][label]
        lines.append(f"| {label} | {result['confusion_matrix'][i][i]}/{row['support']} | {row['precision']:.1%} | {row['recall']:.1%} | {row['f1']:.3f} |")
    lines += ['', '## 混同行列（行＝正解、列＝予測）', '',
              '| 正解 / 予測 | ' + ' | '.join(result['label_order']) + ' |',
              '|---|' + '---:|' * len(result['label_order'])]
    for label, values in zip(result['label_order'], result['confusion_matrix']):
        lines.append('| ' + label + ' | ' + ' | '.join(map(str, values)) + ' |')
    lines += ['', '## 出典別のtest正解数', '']
    for source, row in result['by_source'].items():
        lines.append(f"- {source}: {row['correct']}/{row['n']}（{row['accuracy']:.1%}）")
    lines += ['', '## 適合確認・調整用（testと区別）', '']
    for split in ('train', 'dev'):
        row = metrics[split]
        lines.append(f"- {split}: {row['correct']}/{row['n']}（{row['accuracy']:.1%}）、macro F1 {row['macro_f1']:.3f}")
    lines += ['', '過去・失敗収集素材内の診断であり、未知の実機精度ではありません。少数分類は1件の誤りで大きく変わります。']
    report_text = '\n'.join(lines)
    (report_dir/'report.md').write_text(report_text + '\n')
    display(Markdown(report_text))
    display(Markdown((report_dir/'misclassifications.md').read_text()))

    history = [json.loads(line) for line in (ROOT/'reports'/f"{selected['trial']}-dev-history.jsonl").read_text().splitlines()]
    all_labels = list(manifest['label_to_id'])
    history_lines = ['## epochごとのdev：通常対話16分類', '',
        '| epoch | dev loss | macro F1 | ' + ' | '.join(all_labels) + ' |',
        '|---:|---:|---:|' + '---:|' * len(all_labels)]
    for item in history:
        values = [f"{item['epoch']:g}", f"{item['loss']:.4f}", f"{item['macro_f1']:.3f}"]
        for label in all_labels:
            row = item['per_class'][label]
            values.append(f"{int(row['correct'])}/{int(row['support'])}")
        history_lines.append('| ' + ' | '.join(values) + ' |')
    history_lines += ['', '各分類は正解数/対象数。選択は16分類全体のdev macro F1、同点ならloss・早いepochの順です。']
    history_text = '\n'.join(history_lines)
    (report_dir/'normal-dev-history.md').write_text(history_text + '\n')
    display(Markdown(history_text))

    predictions = {row['id']: row for row in (json.loads(line) for line in (report_dir/'train-predictions.jsonl').read_text().splitlines())}
    batches = [
        ('dev_tools/request_classifier/curation/2026-10-24-normal-dialogue-feedback-1548/candidates.jsonl', 15,
         '実機フィードバック15件', 'feedback-train-check.md'),
        ('dev_tools/request_classifier/curation/2026-10-24-normal-dialogue-tool-contrast/candidates.jsonl', 32,
         '今回追加した道具・雑談の合成例32件', 'contrast-train-check.md'),
    ]
    for source, expected_count, title, filename in batches:
        feedback = [json.loads(line) for line in (ROOT/source).read_text().splitlines()]
        assert len(feedback) == expected_count
        feedback_lines = [f'## {title}（学習内の確認）', '',
            '全件trainに含みます。未知の発話に対する評価ではありません。', '',
            '| 本文 | 正解 | 予測 | 正誤 |', '|---|---|---|---|']
        correct = 0
        for row in feedback:
            prediction = predictions[row['id']]
            assert prediction['gold'] == row['label'] and prediction['text'] == row['text']
            correct += int(prediction['correct'])
            text = row['text'].replace('|', r'\|').replace('\n', '<br>')
            feedback_lines.append(f"| {text} | {row['label']} | {prediction['predicted']} | {'○' if prediction['correct'] else '×'} |")
        feedback_lines += ['', f'学習内の正解：{correct}/{len(feedback)}']
        feedback_text = '\n'.join(feedback_lines)
        (report_dir/filename).write_text(feedback_text + '\n')
        display(Markdown(feedback_text))


## 重み・全予測・誤分類・実行記録を保存

In [ ]:
from google.colab import files
manifest = json.loads((ROOT/'reports/manifest.json').read_text())
assert manifest.get('finished_at_utc')
selected = [ROOT/'weights'/item['path'] for item in manifest['selected_checkpoints'].values()]
paths = sorted(p for p in ROOT.rglob('*') if p.is_file() and (
    p.relative_to(ROOT).parts[0] in {'reports', 'dev_tools'} and '__pycache__' not in p.parts or
    p.relative_to(ROOT).parts[0] == 'weights' and (p.suffix in {'.json', '.jsonl'} or any(p.is_relative_to(s) for s in selected)) or
    p.parent == ROOT and p.suffix in {'.json', '.txt', '.log'} and p.name != 'artifact-sha256.json'))
checksums = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
(ROOT/'artifact-sha256.json').write_text(json.dumps(checksums, indent=2))
archive = Path('/content/modernbert-normal578-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=2) as z:
    for p in paths + [ROOT/'artifact-sha256.json']: z.write(p, p.relative_to(ROOT))
print('保存する結果ZIP:', archive.name, archive.stat().st_size, 'bytes')
print('SHA-256:', hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
